In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
import ast
from scipy.sparse import csr_matrix

In [2]:
output_path = "../input/"
train_path = output_path + "train/"
val_path = output_path + "val/"

train_df = train_path + "train_data_split.csv"
val_df = val_path + "val_data_split.csv"
train_emb = train_path + "train_embeddings_split.npy"
val_emb = val_path + "val_embeddings_split.npy"

y_train_sparse_path = train_path + "Y_train_sparse.npy"
y_val_sparse_path = val_path + "Y_val_sparse.npy"

classes_path = output_path + "classes.npy"
ia_weights_path = "../../general_input/IA.tsv"
final_ia_weights_path = output_path + "ia_weights.npy"

In [3]:
df_train = pd.read_csv(train_df)
df_val = pd.read_csv(val_df)
X_train = np.load(train_emb)
X_val = np.load(val_emb)

In [4]:
def safe_eval_and_flatten(x):
    if pd.notna(x) and x.strip().startswith('['):
        try:
            # 1. Convert string to list of lists (e.g., "[['GO:1'], ['GO:2', 'GO:3']]")
            evaluated_list_of_lists = ast.literal_eval(x)
            
            # 2. Flatten the list of lists into a single list
            return [term for sublist in evaluated_list_of_lists for term in sublist]
        except:
            # Return empty list if evaluation fails
            return []
    return []

def parse_go_terms_flattened(go_terms_series):
    # Apply the safe evaluation and flattening function to the series
    return go_terms_series.apply(safe_eval_and_flatten)

In [5]:
df_train

,db,id,name,full_name,organism,taxon,gen,PE,SV,sequence,length,go_terms,aspect
0,sp,A0A023FT45,EV985_AMBPA,Evasin P985,Amblyomma parvum,251391,NaN,2,1,MHSTIAYVSLLPLALFVAMHGASTDEESEELGASTDVDYEELDANC...,132,[['GO:0019957']],['F']
1,sp,A0A023G6B6,E1180_AMBTT,Evasin P1180,Amblyomma triste,251400,NaN,3,2,MARNWSFRVIFVSAMWCALLKFATLEEPKDGYDYTEGCPFVVLGNG...,116,[['GO:0019957']],['F']
2,sp,A0A023G9N9,E1183_AMBTT,Evasin P1183,Amblyomma triste,251400,NaN,3,2,MTRNWSFRVIFVSAMWCALLKFATLEAPKDDFEYDGGCPFVVLDNG...,115,[['GO:0019957']],['F']
3,sp,A0A023I7E1,ENG1_RHIMI,"Glucan endo-1,3-beta-D-glucosidase 1",Rhizomucor miehei,4839,ENG1,1,1,MRFQVIVAAATITMITSYIPGVASQSTSDGDDLFVPVSNFDPKSIF...,796,"[['GO:0042973'], ['GO:0000272']]","['F', 'P']"
4,sp,A0A023PXP4,YL235_YEAST,Putative uncharacterized protein YLR235C,Saccharomyces cerevisiae (strain ATCC 204508 /...,559292,YLR235C,5,1,MLIGAPSNMRLRGALELLWRRLLHGLMQLRLVLKMHICSQLNHAIK...,132,[['GO:0006974']],['P']
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58356,sp,X2JDY8,RHGF_DROME,Rho GTPase-activating protein Graf,Drosophila melanogaster,7227,Graf,1,1,MGGGKNVRRGLEPLEFEECIVDSPDFRENLNRHEKELDHTSHQIKR...,1025,"[['GO:0140036'], ['GO:0031410'], ['GO:0002092'...","['F', 'C', 'P', 'P', 'C', 'C', 'C', 'F', 'P', ..."
58357,sp,X4Y2L4,LHYAL_HIRNI,Hyaluronoglucuronidase,Hirudo nipponia,42736,NaN,1,1,MKEIAVTIDDKNVIASVSESFHGVAFDASLFSPKGLWSFVDITSPK...,489,"[['GO:0033906'], ['GO:0030214']]","['F', 'P']"
58358,sp,X5JA13,SECAA_ARATH,Exocyst complex component SEC10a,Arabidopsis thaliana,3702,SEC10a,1,1,MTEGIRARGPRSSSVNSVPLILDIEDFKGDFSFDALFGNLVNDLLP...,825,"[['GO:0070062'], ['GO:0005829'], ['GO:0005515'...","['C', 'C', 'F', 'C', 'C', 'C', 'C', 'P']"
58359,sp,X5JB51,SECAB_ARATH,Exocyst complex component SEC10b,Arabidopsis thaliana,3702,SEC10b,1,1,MTERIRARGPRSSSVNSVPLILDIEDFKGDFSFDALFGNLVNDLLP...,829,"[['GO:0070062'], ['GO:0005829'], ['GO:0005886'...","['C', 'C', 'C', 'C', 'C', 'C', 'P', 'F']"


In [6]:
y_train_lists = parse_go_terms_flattened(df_train['go_terms'])
y_val_lists = parse_go_terms_flattened(df_val['go_terms'])

In [7]:
# Binarizer
mlb = MultiLabelBinarizer(sparse_output=True)

In [8]:
# Codification adjustment
Y_train_sparse = mlb.fit_transform(y_train_lists)

In [9]:
Y_val_sparse = mlb.transform(y_val_lists)

/home/maria/miniconda3/envs/torch_env_kaggle_wi/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['GO:0000048', 'GO:0000098', 'GO:0000099', 'GO:0000109', 'GO:0000270', 'GO:0000308', 'GO:0000430', 'GO:0000438', 'GO:0000716', 'GO:0000832', 'GO:0000835', 'GO:0000962', 'GO:0001000', 'GO:0001067', 'GO:0001070', 'GO:0001072', 'GO:0001117', 'GO:0001172', 'GO:0001512', 'GO:0001572', 'GO:0001586', 'GO:0001587', 'GO:0001607', 'GO:0001681', 'GO:0001695', 'GO:0001770', 'GO:0001790', 'GO:0001792', 'GO:0001814', 'GO:0001815', 'GO:0001847', 'GO:0001855', 'GO:0001861', 'GO:0001882', 'GO:0001884', 'GO:0001910', 'GO:0001918', 'GO:0002097', 'GO:0002101', 'GO:0002132', 'GO:0002136', 'GO:0002153', 'GO:0002172', 'GO:0002178', 'GO:0002190', 'GO:0002191', 'GO:0002193', 'GO:0002220', 'GO:0002222', 'GO:0002228', 'GO:0002252', 'GO:0002263', 'GO:0002264', 'GO:0002266', 'GO:0002274', 'GO:0002295', 'GO:0002312', 'GO:0002384', 'GO:0002398', 'GO:0002399', 'GO:0002419', 

In [10]:
print("--- Codification results ---")
print(f"Label shape (Y_train): {Y_train_sparse.shape}")
print(f"Object type: {type(Y_train_sparse)}") 
# <class 'scipy.sparse.csr.csr_matrix'>
print(f"RAM used (aprox): {Y_train_sparse.data.nbytes / 1024**2:.2f} MB")

--- Codification results ---
Label shape (Y_train): (58361, 22828)
Object type: <class 'scipy.sparse._csr.csr_matrix'>
RAM used (aprox): 2.70 MB


In [11]:
# Save Y_train_sparse and Y_val_sparse
np.save(y_train_sparse_path, Y_train_sparse)
np.save(y_val_sparse_path, Y_val_sparse)

# Save classes.npy (used later)

In [12]:
from tqdm import tqdm

In [13]:
# We use a SET to automatically store only unique terms
unique_classes = set()

# Iterate row by row to save RAM
for row in tqdm(df_train['go_terms']):
    terms = safe_eval_and_flatten(row)
    unique_classes.update(terms)

100%|██████████| 58361/58361 [00:01<00:00, 45539.96it/s]


In [14]:
sorted_classes = sorted(list(unique_classes))
classes_array = np.array(sorted_classes)

print(f"Number of unique GO terms in training set: {len(sorted_classes)}")

Number of unique GO terms in training set: 22828


In [15]:
num_cols_matrix = Y_train_sparse.shape[1]
    
if len(classes_array) == num_cols_matrix:
    print("OK! The number of classes matches the trained matrix.")
    
    # Save so we don't need to repeat this ever again
    np.save(classes_path, classes_array)
    print(f"File saved to: {output_path}classes.npy")
else:
    print(f"WARNING: Size mismatch.")
    print(f"Unique terms found in CSV: {len(classes_array)}")
    print(f"Columns in sparse matrix: {num_cols_matrix}")
    print("Check if data filtering was identical.")

OK! The number of classes matches the trained matrix.
File saved to: ../input/classes.npy


# Save IA_weights (used on validation)

In [16]:
df_weights = pd.read_csv(ia_weights_path, sep="\t", names=["go_terms", "ia"])
df_weights.head()

,go_terms,ia
0,GO:0000001,0.000000
1,GO:0000002,2.849666
2,GO:0000011,0.137504
3,GO:0000012,6.038630
4,GO:0000017,0.514573


In [17]:
ia_score_map = df_weights.set_index('go_terms')['ia'].to_dict()

In [18]:
MIN_IA_SCORE = 0.0

In [19]:
LABEL_COUNT = len(mlb.classes_)
LABEL_COUNT

22828

In [20]:
ia_weights_ordered = np.zeros(LABEL_COUNT, dtype=np.float32)

In [21]:
for i, go_term in enumerate(mlb.classes_):
    weight = ia_score_map.get(go_term, MIN_IA_SCORE)
    ia_weights_ordered[i] = weight

In [22]:
ia_weights_ordered.shape

(22828,)

In [23]:
np.save(final_ia_weights_path, ia_weights_ordered)